# Benchmark reproducible de CNN2 — CUDA

Ejecutar únicamente en la computadora NVIDIA. Este notebook nunca simula CUDA ni cae a CPU. El protocolo oficial usa el mismo dataset, checkpoint, seed, batch y número de repeticiones que CPU.

In [ ]:
from pathlib import Path
import hashlib
import platform
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
CHECKPOINT = ROOT / 'results/initialization/cnn2_seed42_initial.pt'

## 1. Hardware y entorno obligatorios

Debe existir `nvidia-smi`, la rueda debe ser `+cu126` y `torch.cuda.is_available()` debe ser verdadero.

In [ ]:
import numpy as np
import torch

subprocess.run(['nvidia-smi'], check=True)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, '| compilado CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert platform.python_version() == '3.11.9'
assert str(torch.__version__) == '2.6.0+cu126'
assert torch.version.cuda == '12.6'
assert np.__version__ == '1.26.4'
assert torch.cuda.is_available(), 'CUDA no está disponible; no continuar.'

## 2. Checkpoint compartido

CUDA debe cargar exactamente el mismo estado inicial que CPU.

In [ ]:
digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest().upper()
print('SHA-256:', digest)
assert digest == 'B5589BFD7B7EF9AF3E1E41ECEE5C42BA4E9BD75A6989A092963324FC4453D19F'

## 3. Sincronización y diferencias inevitables

Cada segmento GPU se rodea con `torch.cuda.synchronize()` y se mide con `time.perf_counter`. Esto evita medir únicamente el lanzamiento asíncrono. CPU y CUDA comparten orden de muestras y distribuciones de augmentation, pero sus RNG/kernels pueden producir valores no idénticos. Se comparan tiempos y métricas, no igualdad bit a bit después de entrenar.

## 4. Smoke test CUDA

Ejecuta solo dos batches de entrenamiento. Si CUDA no está disponible, el comando falla antes de crear resultados.

In [ ]:
command = [
    sys.executable, '-m', 'src.benchmark',
    '--device', 'cuda', '--epochs', '1', '--batch-size', '32',
    '--repetitions', '1', '--num-workers', '0', '--seed', '42',
    '--warmup-batches', '1', '--max-train-batches', '2',
    '--max-eval-batches', '1', '--output-dir', 'results/benchmarks_smoke',
]
subprocess.run(command, cwd=ROOT, check=True)

## 5. Benchmark CUDA oficial — ejecutar después

```powershell
nvidia-smi
python -m src.benchmark --device cuda --epochs 10 --batch-size 32 --repetitions 3 --num-workers 0 --seed 42 --warmup-batches 2 --output-dir results/benchmarks
nvidia-smi
```

El directorio incluirá la captura textual `nvidia-smi.txt`. No publique resultados CUDA si el JSON indica `cuda_available: false` o `full_protocol: false`.